# 04 — Test d'intégration du service de scoring

Ce notebook vérifie que la couche applicative utilise correctement les **artefacts gelés** et le code partagé dans `src/`.

Il ne réentraîne aucun modèle.

### Objectifs
1. charger `FraudScoringService` dans l'environnement `.venv312` ;
2. scorer un petit lot contrôlé de transactions ;
3. vérifier la distinction entre **règle exacte**, **score H2**, **probabilité calibrée** et **décision économique** ;
4. exécuter quelques assertions avant l'intégration Streamlit.


## 1. Chargement du service


In [1]:
import sys
import pandas as pd

from src.service import FraudScoringService

print(sys.executable)

service = FraudScoringService()

print("Service chargé OK")

C:\Users\Emmanuel\Documents\Professionnel\Formations\DIT\fraude-mobile-money\.venv312\Scripts\python.exe
Service chargé OK


### Conclusion

Le notebook pointe vers le Python du projet et `FraudScoringService` se charge correctement. Les contrôles SHA-256 intégrés au service garantissent que les artefacts chargés sont les versions gelées.


## 2. Jeu de transactions contrôlé


In [2]:
sample = pd.DataFrame({
    "step": [
        400,
        400,
        400,
        400,
        400,
    ],

    "type": [
        "TRANSFER",
        "TRANSFER",
        "PAYMENT",
        "CASH_OUT",
        "CASH_IN",
    ],

    "amount": [
        500_000.0,
        10_000_000.0,
        5_000.0,
        100_000.0,
        50_000.0,
    ],

    "oldbalance_org": [
        500_000.0,
        12_000_000.0,
        25_000.0,
        100_000.0,
        10_000.0,
    ],

    "oldbalance_dest": [
        0.0,
        0.0,
        10_000.0,
        50_000.0,
        25_000.0,
    ],
})

sample

,step,type,amount,oldbalance_org,oldbalance_dest
0,400,TRANSFER,500000.0,500000.0,0.0
1,400,TRANSFER,10000000.0,12000000.0,0.0
2,400,PAYMENT,5000.0,25000.0,10000.0
3,400,CASH_OUT,100000.0,100000.0,50000.0
4,400,CASH_IN,50000.0,10000.0,25000.0


Le lot de cinq transactions couvre volontairement plusieurs cas :
- un `TRANSFER` qui vide exactement le compte ;
- un `TRANSFER` atypique de 10 M sans vidage exact ;
- une transaction `PAYMENT` à faible risque ;
- un `CASH_OUT` avec vidage exact ;
- un `CASH_IN` à faible risque.


## 3. Scoring complet


In [3]:
sample_scores = service.score(
    sample,
    investigation_cost=100.0,
    loss_fraction=1.0,
    intervention_effectiveness=0.8,
)

sample_scores

,step,type,amount,oldbalance_org,oldbalance_dest,exact_drain_rule,residual_ml_score,h2_priority_score,p_fraud_calibrated,expected_net_value,economic_review
0,400,TRANSFER,500000.0,500000.0,0.0,True,5.653845e-01,2.565385e+00,9.300158e-01,3.719063e+05,True
1,400,TRANSFER,10000000.0,12000000.0,0.0,False,9.884862e-01,9.884862e-01,9.669265e-01,7.735312e+06,True
2,400,PAYMENT,5000.0,25000.0,10000.0,False,5.024578e-08,5.024578e-08,9.708045e-08,-9.999961e+01,False
3,400,CASH_OUT,100000.0,100000.0,50000.0,True,1.582739e-02,2.015827e+00,7.000380e-02,5.500304e+03,True
4,400,CASH_IN,50000.0,10000.0,25000.0,False,2.736006e-07,2.736006e-07,4.619342e-07,-9.998152e+01,False


In [4]:
sample_scores[
    [
        "type",
        "amount",
        "oldbalance_org",
        "oldbalance_dest",
        "exact_drain_rule",
        "residual_ml_score",
        "h2_priority_score",
        "p_fraud_calibrated",
        "expected_net_value",
        "economic_review",
    ]
]

,type,amount,oldbalance_org,oldbalance_dest,exact_drain_rule,residual_ml_score,h2_priority_score,p_fraud_calibrated,expected_net_value,economic_review
0,TRANSFER,500000.0,500000.0,0.0,True,5.653845e-01,2.565385e+00,9.300158e-01,3.719063e+05,True
1,TRANSFER,10000000.0,12000000.0,0.0,False,9.884862e-01,9.884862e-01,9.669265e-01,7.735312e+06,True
2,PAYMENT,5000.0,25000.0,10000.0,False,5.024578e-08,5.024578e-08,9.708045e-08,-9.999961e+01,False
3,CASH_OUT,100000.0,100000.0,50000.0,True,1.582739e-02,2.015827e+00,7.000380e-02,5.500304e+03,True
4,CASH_IN,50000.0,10000.0,25000.0,False,2.736006e-07,2.736006e-07,4.619342e-07,-9.998152e+01,False


### Lecture des résultats

- Les deux transactions de **vidage exact** reçoivent un score H2 supérieur à 2 et sont donc prioritaires dans un classement par capacité.
- Le `TRANSFER` de **10 000 000** n'est pas un vidage exact, mais le modèle résiduel lui attribue un score très élevé (`≈ 0,9885`) et le moteur probabiliste une probabilité calibrée d'environ **96,7 %**.
- `PAYMENT` et `CASH_IN` reçoivent des risques quasi nuls et une valeur nette attendue négative dans le scénario courant.
- **3 transactions sur 5** sont recommandées pour contrôle économique.

Ce test illustre pourquoi le système sépare :
- **H2** pour le ranking ;
- la **probabilité calibrée** pour la décision économique.


## 4. Assertions d'intégration


In [5]:
assert len(sample_scores) == 5

assert sample_scores[
    "exact_drain_rule"
].sum() == 2

assert (
    sample_scores.loc[
        sample_scores["exact_drain_rule"],
        "h2_priority_score",
    ] > 2.0
).all()

assert (
    sample_scores[
        "p_fraud_calibrated"
    ].between(0, 1)
).all()

print("Test d'intégration du service : OK")

Test d'intégration du service : OK


# Conclusion

Le service reproduit le comportement attendu sur un échantillon contrôlé :
- 5 transactions scorées ;
- 2 vidages exacts ;
- scores H2 cohérents ;
- probabilités bornées entre 0 et 1 ;
- décisions économiques cohérentes.

Le backend de scoring est donc prêt à être consommé par l'interface Streamlit sans duplication de logique ML.
